### Configuration

In [ ]:
from glob import glob
from pathlib import Path

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt, ticker
from matplotlib.lines import Line2D

In [ ]:
DB_ORDER = ['CockroachDB', 'TiDB', 'Citus']
DB_LABELS = {'COCKROACH': 'CockroachDB', 'TIDB': 'TiDB', 'CITUS': 'Citus'}

COLORS = {'CockroachDB': '#4C72B0', 'TiDB': '#DD8452', 'Citus': '#55A868'}
MARKERS = {'CockroachDB': 'o', 'TiDB': 's', 'Citus': '^'}
LINESTYLES = {'CockroachDB': '-', 'TiDB': '--', 'Citus': ':'}

NODE_COUNTS = [3, 5, 10, 20]

GROUPS = {
    1: {
        'label': 'Group 1 — Scan-Dominated (Q1, Q6, Q14, Q19)',
        'queries': ['Q01', 'Q06', 'Q14', 'Q19']
    },
    2: {
        'label': 'Group 2 — Join-Dominated (Q3, Q5, Q7, Q8, Q9, Q10)',
        'queries': ['Q03', 'Q05', 'Q07', 'Q08', 'Q09', 'Q10']
    },
    3: {
        'label': 'Group 3 — Subquery-Driven (Q2, Q11, Q17, Q22)',
        'queries': ['Q02', 'Q11', 'Q17', 'Q22']
    },
    4: {
        'label': 'Group 4 — Multi-Stage (Q4, Q12, Q13, Q15, Q16, Q18, Q20, Q21)',
        'queries': ['Q04', 'Q12', 'Q13', 'Q15', 'Q16', 'Q18', 'Q20', 'Q21']
    },
}
ALL_QUERIES = [f'Q0{i}' if i < 10 else f'Q{i}' for i in range(1, 23)]

FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

In [ ]:
def load_data(csv_path: list['str']) -> pd.DataFrame:
    frames = []
    for path in csv_path:
        df = pd.read_csv(path)
        frames.append(df)

    raw = pd.concat(frames, ignore_index=True)

    raw['database'] = raw['database'].map(DB_LABELS)
    raw['query_name'] = raw['query_name'].str.upper().map(lambda s: "Q" + s[1:].zfill(2) if s.startswith("Q") else s)
    raw['node_count'] = raw['node_count'].astype(int)
    raw['duration_ms'] = pd.to_numeric(raw['duration_ms'], errors='coerce')

    before = len(raw)
    raw = raw.dropna(subset=['duration_ms'])
    if len(raw) < before:
        print(f'Dropped {before - len(raw)} rows with unparseable duration_ms.')

    return raw

In [ ]:
csv_paths = glob('../*.csv')
raw = load_data(csv_paths)

### Aggregation

In [ ]:
def compute_stats(df: pd.DataFrame) -> pd.DataFrame:
    group = df.groupby(['database', 'node_count', 'query_name'])['duration_ms']
    stats = group.agg(
        median_ms='median',
        mean_ms='mean',
        std_ms='std',
        min_ms='min',
        max_ms='max',
        n_runs='count'
    ).reset_index()

    baseline = (
        stats[stats['node_count'] == 3]
        [['database', 'query_name', 'median_ms']]
        .rename(columns={'median_ms': 'baseline_ms'})
    )

    stats = stats.merge(baseline, on=['database', 'query_name'], how='left')
    stats['speedup'] = stats['baseline_ms'] / stats['median_ms']

    return stats


stats = compute_stats(raw)

### Plotting

In [ ]:
def _db_legend_handles():
    return [
        Line2D([0], [0], color=COLORS[db], marker=MARKERS[db],
               linestyle=LINESTYLES[db], linewidth=1.8,
               markersize=7, label=db)
        for db in DB_ORDER if db in COLORS
    ]


def _save_figure(fig, filename: str) -> None:
    path = FIG_DIR / filename
    fig.savefig(path, bbox_inches='tight', pad_inches=0.2)


def plot_latency_vs_nodes(stats: pd.DataFrame, queries: list[str], title: str, filename: str, ncols: int = 3) -> None:
    nrows = int(np.ceil(len(queries) / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5.5 * ncols, 4 * nrows),
                             sharey=False)
    axes = np.array(axes).flatten()

    for idx, query in enumerate(queries):
        ax = axes[idx]
        subset = stats[stats['query_name'] == query]
        for db in DB_ORDER:
            db_data = subset[subset['database'] == db].sort_values('node_count')
            y = db_data['median_ms'].values / 1000.0  # → seconds
            x = db_data['node_count'].values

            mask = ~np.isnan(y)
            ax.plot(x[mask], y[mask],
                    color=COLORS[db], marker=MARKERS[db],
                    linestyle=LINESTYLES[db], linewidth=1.8,
                    markersize=7, label=db)

        ax.set_title(query.upper(), fontweight='bold')
        ax.set_xlabel('Număr de noduri')
        ax.set_ylabel('Latență mediană (s)')
        ax.set_xticks(NODE_COUNTS)
        ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f'))

    fig.legend(handles=_db_legend_handles(),
               loc='lower center', ncol=3,
               bbox_to_anchor=(0.5, 0.01), frameon=False)
    fig.suptitle(title, fontsize=13, fontweight='bold', y=0.98)
    fig.tight_layout(rect=(0, 0.06, 1, 0.94))

    _save_figure(fig, filename)
    plt.show()


def plot_speedup(stats: pd.DataFrame, queries: list[str], title: str, filename: str, ncols: int = 3) -> None:
    nrows = int(np.ceil(len(queries) / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5.5 * ncols, 4 * nrows),
                             sharey=False)
    axes = np.array(axes).flatten()

    for idx, query in enumerate(queries):
        ax = axes[idx]
        subset = stats[stats['query_name'] == query]
        for db in DB_ORDER:
            db_data = subset[subset['database'] == db].sort_values('node_count')
            y = db_data['speedup'].values
            x = db_data['node_count'].values

            mask = ~np.isnan(y)
            ax.plot(x[mask], y[mask],
                    color=COLORS[db], marker=MARKERS[db],
                    linestyle=LINESTYLES[db], linewidth=1.8,
                    markersize=7)

        ax.axhline(1.0, color='grey', linewidth=0.8, linestyle='--')
        ax.set_title(query.upper(), fontweight='bold')
        ax.set_xlabel('Număr de noduri')
        ax.set_ylabel('Accelerație (x)')
        ax.set_xticks(NODE_COUNTS)

    fig.legend(handles=_db_legend_handles(),
               loc='lower center', ncol=3,
               bbox_to_anchor=(0.5, 0.01), frameon=False)
    fig.suptitle(title, fontsize=13, fontweight='bold', y=0.98)
    fig.tight_layout(rect=(0, 0.06, 1, 0.94))

    _save_figure(fig, filename)
    plt.show()


def plot_heatmap(stats: pd.DataFrame, queries: list[str], metric: str, title: str, filename: str) -> None:
    fig, axes = plt.subplots(1, len(DB_ORDER),
                             figsize=(5 * len(DB_ORDER), 0.5 * len(queries) + 2),
                             sharey=True)
    vmax = (stats[stats['query_name'].isin(queries)][metric].max()) / 1000.0

    for ax, db in zip(axes, DB_ORDER):
        pivot = (
                    stats[(stats['database'] == db) & (stats['query_name'].isin(queries))]
                    .pivot(index='query_name', columns='node_count', values=metric)
                    .reindex(index=queries, columns=NODE_COUNTS)
                ) / 1000.0

        im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd',
                       vmin=0, vmax=vmax)
        ax.set_xticks(range(len(NODE_COUNTS)))
        ax.set_xticklabels(NODE_COUNTS)
        ax.set_yticks(range(len(queries)))
        ax.set_yticklabels([q.upper() for q in queries])
        ax.set_title(db, fontweight='bold')
        ax.set_xlabel('Număr de noduri')

        for r in range(pivot.shape[0]):
            for c in range(pivot.shape[1]):
                val = pivot.values[r, c]
                if not np.isnan(val):
                    ax.text(c, r, f'{val:.1f}', ha='center', va='center',
                            fontsize=8,
                            color='white' if val > 0.6 * vmax else 'black')
                else:
                    ax.text(c, r, 'N/A', ha='center', va='center',
                            fontsize=8, color='grey')

    plt.colorbar(im, ax=axes[-1], label='Median latency (s)')

    fig.suptitle(title, fontsize=13, fontweight='bold', y=0.98)
    fig.tight_layout(rect=(0, 0, 1, 0.94))

    _save_figure(fig, filename)
    plt.show()


def plot_boxplots(raw: pd.DataFrame, queries: list[str], node_count: int, title: str, filename: str,
                  ncols: int = 3) -> None:
    subset = raw[raw['node_count'] == node_count]
    nrows = int(np.ceil(len(queries) / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(5.5 * ncols, 4 * nrows))
    axes = np.array(axes).flatten()

    for idx, query in enumerate(queries):
        ax = axes[idx]
        q_data = subset[subset['query_name'] == query]
        data_by_db = [
            q_data[q_data['database'] == db]['duration_ms'].values / 1000.0
            for db in DB_ORDER
        ]
        bp = ax.boxplot(
            [d for d in data_by_db if len(d) > 0],
            tick_labels=[db for db, d in zip(DB_ORDER, data_by_db) if len(d) > 0],
            patch_artist=True,
            medianprops=dict(color='black'),
        )
        for patch, db in zip(bp['boxes'],
                             [db for db, d in zip(DB_ORDER, data_by_db) if len(d) > 0]):
            patch.set_facecolor(COLORS[db])

        ax.set_title(query.upper(), fontweight='bold')
        ax.set_ylabel('Timp (s)')
        ax.tick_params(axis='x', rotation=15)
        plt.yscale('log')

    for ax in axes[len(queries):]:
        ax.set_visible(False)

    fig.suptitle(f'{title} - {node_count} noduri', fontsize=13, fontweight='bold', y=0.98)
    fig.tight_layout(rect=(0, 0, 1, 0.94))

    _save_figure(fig, filename)
    plt.show()


def plot_grouped_bar(stats: pd.DataFrame, queries: list[str], node_count: int, title: str, filename: str) -> None:
    subset = stats[stats['node_count'] == node_count]
    x = np.arange(len(queries))
    width = 0.25
    fig, ax = plt.subplots(figsize=(max(10, len(queries) * 1.2), 5))

    for i, db in enumerate(DB_ORDER):
        vals = [
            subset[(subset['database'] == db) &
                   (subset['query_name'] == q)]['median_ms'].values
            for q in queries
        ]
        heights = [v[0] / 1000.0 if len(v) > 0 and not np.isnan(v[0]) else 0
                   for v in vals]
        ax.bar(x + i * width, heights, width,
               label=db, color=COLORS[db], alpha=0.85)

        for j, (h, v) in enumerate(zip(heights, vals)):
            if len(v) == 0 or np.isnan(v[0]):
                ax.text(x[j] + i * width, 0.1, 'N/A',
                        ha='center', va='bottom', fontsize=7, color='grey')

    ax.set_xticks(x + width)
    ax.set_xticklabels([q.upper() for q in queries])
    ax.set_ylabel('Latență mediană (s)')
    ax.set_title(f'{title} — {node_count} noduri', fontweight='bold')
    ax.legend(frameon=False)

    fig.tight_layout()

    _save_figure(fig, filename)
    plt.show()

### Grupa 1 - Interogări dominate de scanări

In [ ]:
QG1 = GROUPS[1]['queries']

plot_latency_vs_nodes(
    stats, QG1,
    title='Grupa 1 - Interogări dominate de scanări: Latență mediană vs Număr de noduri',
    filename='g1_latency_vs_nodes.png',
    ncols=2
)

In [ ]:
plot_speedup(
    stats, QG1,
    title='Grupa 1 - Interogări dominate de scanări: Accelerație relativă (linie de bază: 3 noduri)',
    filename='g1_speedup.png',
    ncols=2
)

In [ ]:
plot_heatmap(
    stats, QG1,
    metric='median_ms',
    title='Grupa 1 - Interogări dominate de scanări: Latență mediană - Hartă termică (s)',
    filename='g1_heatmap.png'
)

In [ ]:
plot_boxplots(
    raw, QG1, node_count=10,
    title='Grupa 1 - Interogări dominate de scanări: Distribuția execuțiilor',
    filename='g1_boxplots_10n.png',
    ncols=2
)

### Grupa 2 - Interogări dominate de joins

In [ ]:
QG2 = GROUPS[2]['queries']

plot_latency_vs_nodes(
    stats, QG2,
    title='Group 2 - Interogări dominate de îmbinări: Latență mediană vs. Număr de noduri',
    filename='g2_latency_vs_nodes.png',
    ncols=3
)

In [ ]:
plot_speedup(
    stats, QG2,
    title='Grupa 2 - Interogări dominate de îmbinări: Accelerație relativă (linie de bază: 3 noduri)',
    filename='g2_speedup.png',
    ncols=3
)

In [ ]:
plot_heatmap(
    stats, QG2,
    metric='median_ms',
    title='Grupa 2 - Interogări dominate de îmbinări: Latență mediană - Hartă termică (s)',
    filename='g2_heatmap.png'
)

In [ ]:
plot_boxplots(
    raw, QG2, node_count=10,
    title='Grupa 2 - Interogări dominate de îmbinări: Distribuția execuțiilor',
    filename='g2_boxplots_10n.png',
    ncols=3
)

### Grupa 3

In [ ]:
QG3 = GROUPS[3]['queries']

plot_latency_vs_nodes(
    stats, QG3,
    title='Grupa 3 - Interogări conduse de subinterogări: Latență mediană vs. Număr de noduri',
    filename='g3_latency_vs_nodes.png',
    ncols=2
)

In [ ]:
plot_speedup(
    stats, QG3,
    title='Grupa 3 - Interogări conduse de subinterogări: Accelerație relativă la linia de bază cu 3 noduri',
    filename='g3_speedup.png',
    ncols=2
)

In [ ]:
plot_heatmap(
    stats, QG3,
    metric='median_ms',
    title='Grupa 3 - Interogări conduse de subinterogări: Hartă termică a latenței mediane (s)',
    filename='g3_heatmap.png'
)

In [ ]:
plot_boxplots(
    raw, QG3, node_count=10,
    title='Grupa 3 - Interogări conduse de subinterogări: Distribuția rulărilor',
    filename='g3_boxplots_10n.png',
    ncols=2
)

### Grupa 4

In [ ]:
QG4 = GROUPS[4]['queries']

plot_latency_vs_nodes(
    stats, QG4,
    title='Grupa 4 - Interogări în mai multe etape: Latență mediană vs. Număr de noduri',
    filename='g4_latency_vs_nodes.png',
    ncols=4
)

In [ ]:
plot_speedup(
    stats, QG4,
    title='Grupa 4 - Interogări în mai multe etape: Accelerație relativă la linia de bază cu 3 noduri',
    filename='g4_speedup.png',
    ncols=4
)

In [ ]:
plot_heatmap(
    stats, QG4,
    metric='median_ms',
    title='Grupa 4 - Interogări în mai multe etape: Hartă termică a lateței mediane (s)',
    filename='g4_heatmap.png'
)

In [ ]:
plot_boxplots(
    raw, QG4, node_count=10,
    title='Grupa 4 - Interogări în mai multe etape: Distribuția rulărilor',
    filename='g4_boxplots_10n.png',
    ncols=4
)

### Sinteză între grupe

In [ ]:
plot_heatmap(
    stats, ALL_QUERIES,
    metric='median_ms',
    title='Toate interogările - Latență Mediană - Hartă termică (s)',
    filename='global_heatmap.png'
)

In [ ]:
plot_grouped_bar(
    stats, ALL_QUERIES, node_count=3,
    title='Toate interogările - Latență mediană 3 noduri',
    filename='global_grouped_bar_3n.png'
)

plot_grouped_bar(
    stats, ALL_QUERIES, node_count=5,
    title='Toate interogările - Latență mediană 5 noduri',
    filename='global_grouped_bar_5n.png'
)

plot_grouped_bar(
    stats, ALL_QUERIES, node_count=10,
    title='Toate interogările - Latență mediană 10 noduri',
    filename='global_grouped_bar_10n.png'
)

plot_grouped_bar(
    stats, ALL_QUERIES, node_count=20,
    title='Toate interogările - Latență mediană 20 noduri',
    filename='global_grouped_bar_20n.png'
)

In [ ]:
stats['cv'] = stats['std_ms'] / stats['mean_ms']

fig, ax = plt.subplots(figsize=(10, 4))
for db in DB_ORDER:
    subset = stats[stats['database'] == db].groupby('node_count')['cv'].median()
    ax.plot(subset.index, subset.values,
            color=COLORS[db], marker=MARKERS[db],
            linestyle=LINESTYLES[db], linewidth=1.8, markersize=7, label=db)

ax.set_xlabel('Număr de noduri')
ax.set_ylabel('Coeficient de variație median')
ax.set_title('Variabilitate CV median vs. Număr de noduri', fontweight='bold')
ax.set_xticks(NODE_COUNTS)
ax.legend(frameon=False)
fig.tight_layout()
_save_figure(fig, 'global_cv_vs_nodes.png')
plt.show()

In [ ]:
speedup_summary = (
    stats[stats['node_count'] == 20]
    .groupby('database')['speedup']
    .agg(['mean', 'median', 'min', 'max'])
    .round(2)
)
print(speedup_summary)

In [ ]:
pivot = stats.pivot_table(
    index='query_name',
    columns=['database', 'node_count'],
    values='median_ms',
)

pivot = pivot / 1000.0

pivot = pivot.reindex(index=ALL_QUERIES)
pivot = pivot.reindex(columns=pd.MultiIndex.from_product(
    [DB_ORDER, NODE_COUNTS], names=['System', 'Nodes']
))

print(pivot.head())
pivot.to_csv(FIG_DIR / 'ok.csv', float_format='%.3f')